In [ ]:
"""
Assignment 2: Analyzing and Implementing Divide-and-Conquer Algorithms

This script compares Merge Sort and Quick Sort on three kinds of input:
1. Sorted data
2. Reverse-sorted data
3. Random data

It records:
- Execution time
- Peak memory usage

Notes:
- Quick Sort here uses the LAST element as the pivot.
  This is intentional because it helps show the classic worst-case behavior
  on already sorted and reverse-sorted lists.
- Merge Sort uses extra memory to create left/right sublists and merge them.
"""

import random
import time
import tracemalloc
import csv
import sys
from pathlib import Path

# Increase recursion limit so the recursive Quick Sort worst case
# can still run on moderately sized sorted inputs.
sys.setrecursionlimit(30000)



# Merge Sort

def merge_sort(arr):
    """
    Sort a list using Merge Sort and return a new sorted list.

    Why this algorithm is divide-and-conquer:
    - Divide: split the list into two halves
    - Conquer: recursively sort each half
    - Combine: merge the two sorted halves
    """
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    # Recursively sort both halves
    left_half = merge_sort(arr[:mid])
    right_half = merge_sort(arr[mid:])

    # Merge the two sorted halves
    return merge(left_half, right_half)


def merge(left, right):
    """
    Merge two already-sorted lists into one sorted list.
    """
    merged = []
    i = 0  # pointer for left list
    j = 0  # pointer for right list

    # Compare elements from both lists and append the smaller one
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Add the remaining elements, if any
    merged.extend(left[i:])
    merged.extend(right[j:])

    return merged



# Quick Sort

def quick_sort(arr):
    """
    Sort a list using Quick Sort and return the same list after sorting.

    Why this algorithm is divide-and-conquer:
    - Divide: choose a pivot and partition the list around it
    - Conquer: recursively sort the left and right partitions
    - Combine: no explicit merge step is needed
    """
    quick_sort_recursive(arr, 0, len(arr) - 1)
    return arr


def quick_sort_recursive(arr, low, high):
    """
    Recursive helper for Quick Sort.
    """
    if low < high:
        # Partition the array and get the final pivot position
        pivot_index = partition(arr, low, high)

        # Recursively sort elements before and after the pivot
        quick_sort_recursive(arr, low, pivot_index - 1)
        quick_sort_recursive(arr, pivot_index + 1, high)


def partition(arr, low, high):
    """
    Partition the array using the LAST element as pivot.

    After partitioning:
    - elements smaller than or equal to the pivot are on the left
    - elements greater than the pivot are on the right
    """
    pivot = arr[high]
    i = low - 1

    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            # Swap arr[i] and arr[j]
            arr[i], arr[j] = arr[j], arr[i]

    # Place the pivot in its correct sorted position
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1



# Dataset generation

def generate_dataset(size, data_type, seed=42):
    """
    Create a dataset of the requested type and size.

    Parameters:
        size (int): number of elements
        data_type (str): 'sorted', 'reverse_sorted', or 'random'
        seed (int): random seed for reproducibility
    """
    if data_type == "sorted":
        return list(range(size))

    if data_type == "reverse_sorted":
        return list(range(size, 0, -1))

    if data_type == "random":
        rng = random.Random(seed)
        return [rng.randint(1, 100000) for _ in range(size)]

    raise ValueError("data_type must be 'sorted', 'reverse_sorted', or 'random'")



# Performance measurement

def measure_performance(sort_function, data):
    """
    Measure execution time and peak memory usage for a sorting function.

    Returns:
        execution_time_seconds (float)
        peak_memory_kb (float)
    """

    # This helps avoid one-time startup effects from affecting the first reading.
    warmup_arr = [3, 1, 2]
    sort_function(warmup_arr)

    # Work on a copy so the original list is unchanged
    arr = data.copy()

    tracemalloc.start()
    start_time = time.perf_counter()

    sort_function(arr)

    end_time = time.perf_counter()
    current_memory, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    execution_time_seconds = end_time - start_time
    peak_memory_kb = peak_memory / 1024

    return execution_time_seconds, peak_memory_kb



# Experiment runner

def run_experiments():
    """
    Run experiments across multiple input sizes and dataset types.
    """
    sizes = [500, 1000, 1500]
    dataset_types = ["sorted", "reverse_sorted", "random"]

    results = []

    for size in sizes:
        for dataset_type in dataset_types:
            data = generate_dataset(size, dataset_type)

            merge_time, merge_memory = measure_performance(merge_sort, data)
            quick_time, quick_memory = measure_performance(quick_sort, data)

            results.append({
                "size": size,
                "data_type": dataset_type,
                "merge_time_s": merge_time,
                "merge_memory_kb": merge_memory,
                "quick_time_s": quick_time,
                "quick_memory_kb": quick_memory
            })

    return results



# Save results to CSV

def save_results_to_csv(results, output_file):
    """
    Save experiment results to a CSV file.
    """
    fieldnames = [
        "size",
        "data_type",
        "merge_time_s",
        "merge_memory_kb",
        "quick_time_s",
        "quick_memory_kb"
    ]

    with open(output_file, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)



# Main block

if __name__ == "__main__":
    output_csv = Path("sorting_results.csv")

    results = run_experiments()
    save_results_to_csv(results, output_csv)

    print("Results saved to:", output_csv.resolve())
    print()

    for row in results:
        print(
            f"Size={row['size']:>5} | "
            f"Type={row['data_type']:<14} | "
            f"Merge: {row['merge_time_s']:.6f}s, {row['merge_memory_kb']:.2f} KB | "
            f"Quick: {row['quick_time_s']:.6f}s, {row['quick_memory_kb']:.2f} KB"
        )


Results saved to: /content/sorting_results.csv

Size=  500 | Type=sorted         | Merge: 0.003354s, 10.62 KB | Quick: 0.119439s, 15.27 KB
Size=  500 | Type=reverse_sorted | Merge: 0.003112s, 9.80 KB | Quick: 0.298644s, 15.27 KB
Size=  500 | Type=random         | Merge: 0.005115s, 8.30 KB | Quick: 0.008678s, 1.02 KB
Size= 1000 | Type=sorted         | Merge: 0.011270s, 19.59 KB | Quick: 2.858449s, 46.52 KB
Size= 1000 | Type=reverse_sorted | Merge: 0.013340s, 19.59 KB | Quick: 3.727574s, 46.49 KB
Size= 1000 | Type=random         | Merge: 0.017465s, 16.84 KB | Quick: 0.026007s, 1.41 KB
Size= 1500 | Type=sorted         | Merge: 0.023020s, 29.39 KB | Quick: 9.252808s, 77.77 KB
Size= 1500 | Type=reverse_sorted | Merge: 0.012841s, 29.39 KB | Quick: 11.172177s, 77.74 KB
Size= 1500 | Type=random         | Merge: 0.031693s, 24.38 KB | Quick: 0.044705s, 1.34 KB
